# 答案评测：用 deepeval（LLM-as-judge）逐题判定 `batch_qa` 的答案是否正确

读取 `notebooks/batch_qa.ipynb` 产出的 `results.jsonl`，用 deepeval 的 `GEval` 指标（`答案正确性`）让裁判模型对照 **期望答案** 逐题打分（0~1，`≥ PASS_THRESHOLD` 判为通过），输出逐题结果与准确率。

- **只评答案正确性**（问题 + 期望答案 + 实际回答），不做 Faithfulness：`results.jsonl` 里只有读取过的页码（`pages_read` / `tool_calls` 的参数），没有检索到的页面原文。
- 没有 `expected` 的题、`batch_qa` 没成功的题（出错、`skipped_no_text_layer` 等）不送裁判，标为 `not_evaluated` 并写明原因。
- **断点续跑**：`eval_results.jsonl` 里已评过（且答案没变）的题直接跳过；每评完一题立刻追加写盘。评测出错（`error`）的题下次重跑。
- 不依赖 `superindex` 包（只读 jsonl 文件），可以单独导入 Databricks 运行。

| | Databricks | 本地（仓库内） |
|---|---|---|
| 安装 | 运行「环境安装」两个 cell（`%pip install` + 重启 Python） | deepeval **不在项目依赖里**，启动时用 `uv run --group notebook --with deepeval jupyter ...` 临时叠加；**不要运行**「环境安装」两个 cell |
| 裁判模型 | 「模型配置」cell 的 `LLM_CONFIG`（`OPENAI_MODEL` / `OPENAI_BASE_URL` / `OPENAI_API_KEY`），密钥从 `dbutils.secrets` 读 | 仓库根目录 `.env`（与 `batch_qa` 共用同一组配置），也可用「模型配置」cell |
| 输入 | `RESULTS_PATH` 指向 `results.jsonl`（如 `/Volumes/...`） | 默认自动找 `<ROOT_DIR>/results/notebook/<题集名>/runs/<最新时间戳>/results.jsonl`；设了 `NB_WORK_DIR`（与 `batch_qa` 相同，可写进 `.env`）时改从 `<NB_WORK_DIR>/runs/` 下找最新的 |

裁判模型走 **OpenAI 兼容的 Chat Completions** 接口（`openai` 包），配置与 `batch_qa` 相同；网关要求自定义请求头时填 `LLM_EXTRA_HEADERS`。想用另一个（更强的）模型当裁判，设配置项 `JUDGE_MODEL`（环境变量 `SI_NB_JUDGE_MODEL`）。

**产出物**（写在输入文件同目录的 `eval/` 下）

| 文件 | 内容 |
|---|---|
| `eval_results.jsonl` | 逐题：`id`、`question`、`expected`、`answer`、`doc`、`eval_score`、`eval_pass`、`eval_reason`、`eval_status`（`evaluated` / `not_evaluated` / `error`）、`judge_model`、`elapsed_s` |
| `eval_summary.md` | 准确率、未评测数、报错数、逐题表格 |
| `eval.csv`、`eval.xlsx` | 同上的表格（csv 为 utf-8-sig；缺 `openpyxl` 时跳过 xlsx 并提示） |

**命令行执行（本地）**：和 `batch_qa` 一样必须跳过「环境安装」cell：

```bash
uv run --group notebook --with deepeval jupyter nbconvert --to notebook --execute notebooks/eval_deepeval.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

（配置项可用 `SI_NB_*` 环境变量覆盖，见配置 cell，如 `SI_NB_RESULTS=<results.jsonl 路径>`、`SI_NB_JUDGE_CONCURRENCY=8`。）

## 环境安装（仅 Databricks）

**只在 Databricks 上运行下面两个 cell**（已打上 cell tag `databricks-setup`）。本地用 `uv run --group notebook --with deepeval jupyter ...` 启动（deepeval 不进项目依赖，`--with` 只临时叠加、不改 `uv.lock`），**不要运行**这两个 cell。

- 第一个 cell 只放 `%pip install`（Databricks 要求 `%pip` 写在 cell 开头）。默认从 PyPI 装；集群访问不了公网 PyPI 时，把那一行整行换成下面的备选写法（cell 里只保留一行 `%pip`，不要加注释行）。
- 第二个 cell 重启 Python，让新装的包生效；只在 Databricks 上才会真的重启，其它环境只打印一句提示。

备选安装写法（替换第一个 cell 的内容）：

```
%pip install --index-url https://<公司内部 PyPI 镜像>/simple deepeval openai pandas openpyxl python-dotenv
```

In [ ]:
%pip install deepeval openai pandas openpyxl python-dotenv

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# ───────────── 模型配置（必须在 import deepeval 之前运行）─────────────
# 本 cell 只写环境变量。默认不改动任何模型变量：本地有 .env 时行为不变（.env 在下一格"环境"里才加载，且不覆盖这里已设的变量）。
import os

# deepeval：关闭匿名遥测与版本更新提示；不让它自己去读 cwd 下的 .env（下一格统一从 ROOT_DIR/.env 读）
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "1")
os.environ.setdefault("DEEPEVAL_UPDATE_WARNING_OPT_IN", "0")
os.environ.setdefault("DEEPEVAL_DISABLE_DOTENV", "1")

# 想改哪项就取消注释并填值；值为空串的项会被跳过。**不要把密钥明文写进 Notebook**。
# 在 Databricks 上从 secrets 读：dbutils.secrets.get(scope="<scope>", key="<key>")
LLM_CONFIG: dict[str, str] = {
    # ── OpenAI 兼容端点（vLLM、公司网关等）：端点上的模型名，地址一般带 /v1 ──
    # "OPENAI_MODEL": "<模型名>",
    # "OPENAI_BASE_URL": "https://<公司网关>/v1",
    # "OPENAI_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # 等价写法（与 batch_qa 的 SUPERINDEX_* 共用；只认 openai/ 前缀的模型名）：
    # "SUPERINDEX_CHAT_MODEL": "openai/<模型名>",
    # "SUPERINDEX_BASE_URL": "https://<公司网关>/v1",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),
}

# 公司网关要求的自定义请求头（如 {"X-Gateway-Token": dbutils.secrets.get(...)}）。默认空 = 不加。
LLM_EXTRA_HEADERS: dict[str, str] = {}

for _name, _value in LLM_CONFIG.items():
    if _value:
        os.environ[_name] = _value
print("模型配置：写入环境变量 " + (", ".join(k for k, v in LLM_CONFIG.items() if v) or "（无，沿用 .env / 进程环境变量）"))
print("自定义请求头：" + (", ".join(LLM_EXTRA_HEADERS) or "（无）"))

In [ ]:
# 确定唯一的绝对路径基准 ROOT_DIR（.env、results/、相对路径配置都以它为基准，不依赖 cwd、不 chdir）。
# 顺序：环境变量 SI_NB_ROOT_DIR > 从 cwd 向上找到的仓库根 > 当前工作目录。
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path | None:
    """向上找含 pyproject.toml 和 superindex/ 包目录的目录（即本仓库根）；找不到返回 None。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir():
            return folder
    return None


IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
_repo_root = find_repo_root(Path.cwd().resolve())
ROOT_DIR = Path(os.environ.get("SI_NB_ROOT_DIR", "").strip() or _repo_root or Path.cwd()).expanduser().resolve()


def under_root(p: str | Path) -> Path:
    """绝对路径原样（resolve）；相对路径以 ROOT_DIR 为基准。"""
    p = Path(p).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


ENV_FILES = [ROOT_DIR / ".env"] if (ROOT_DIR / ".env").is_file() else []
for _env_file in ENV_FILES:
    load_dotenv(_env_file, override=False, encoding="utf-8")  # 读取 ROOT_DIR/.env；已设置的环境变量优先

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"运行环境: {'Databricks' if IS_DATABRICKS else '本地'}")
print(f"仓库根目录: {_repo_root}" if _repo_root else "未从当前工作目录向上找到仓库根目录")
print(f".env      : {', '.join(map(str, ENV_FILES)) or '（未找到，仅使用进程环境变量）'}")

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用环境变量 SI_NB_<名字> 覆盖（方便命令行 nbconvert 执行），例如 SI_NB_JUDGE_CONCURRENCY=8。
def _env(name: str, default: Any, cast=str) -> Any:
    raw = os.environ.get(f"SI_NB_{name}")
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


def _bool(text: str) -> bool:
    return text.lower() in ("1", "true", "yes", "on")


def _optional_float(text: str) -> float | None:
    return None if text.lower() in ("none", "null", "off") else float(text)


# --- 输入 ---
# batch_qa 产出的 results.jsonl（也可以填它所在的运行目录）。优先读环境变量 SI_NB_RESULTS（或 SI_NB_EVAL_INPUT）。
# 留空 = 自动定位：环境变量 NB_WORK_DIR 非空（与 batch_qa 相同，可写进 .env）时取 <NB_WORK_DIR>/runs/<最新时间戳>/results.jsonl；
# 否则 <ROOT_DIR>/results/notebook/<题集名>/runs/<最新时间戳>/results.jsonl，题集名取自 DATASET_PATH
# （与 batch_qa 一样从 .env 的 DATASET_PATH 或 SI_NB_DATASET 读）；DATASET_PATH 也没有时取
# <ROOT_DIR>/results/notebook/*/runs/*/results.jsonl 里修改时间最新的一个。相对路径以 ROOT_DIR 为基准。
# Databricks 上如 "/Volumes/<catalog>/<schema>/<volume>/superindex_nb/<题集名>/runs/<时间戳>/results.jsonl"
RESULTS_PATH = _env("RESULTS", "") or _env("EVAL_INPUT", "")

# --- 裁判 ---
JUDGE_MODEL = _env("JUDGE_MODEL", "")             # 裁判模型名；空 = 用 OPENAI_MODEL（或 SUPERINDEX_CHAT_MODEL 去掉 openai/ 前缀）
PASS_THRESHOLD = _env("PASS_THRESHOLD", 0.5, float)  # GEval 得分（0~1）≥ 它判为通过
JUDGE_CONCURRENCY = _env("JUDGE_CONCURRENCY", 4, int)  # 同时评测的题数
JUDGE_TIMEOUT = _env("JUDGE_TIMEOUT", 120.0, float)    # 单次裁判请求超时（秒）
JUDGE_TEMPERATURE: float | None = _env("JUDGE_TEMPERATURE", 0.0, _optional_float)  # 模型不接受 temperature 时设 none（不传）
# 仅供自测：SI_NB_JUDGE_FAKE=1 时用不联网的假裁判（按"期望答案里的数字 / 文字是否出现在回答里"打分），走通整条 deepeval 链路
JUDGE_FAKE = _env("JUDGE_FAKE", False, _bool)


def locate_results() -> Path:
    if RESULTS_PATH:
        path = under_root(RESULTS_PATH)
        return path / "results.jsonl" if path.is_dir() else path
    nb_work_dir = os.environ.get("NB_WORK_DIR", "").strip()
    if nb_work_dir:
        work_dir = under_root(nb_work_dir)
        runs = sorted(work_dir.glob("runs/*/results.jsonl"))  # 目录名是时间戳，按名字排序即按时间
        if not runs:
            raise FileNotFoundError(f"NB_WORK_DIR 指向的 {work_dir / 'runs'} 下没有 results.jsonl。"
                                    "先运行 batch_qa，或设 RESULTS_PATH / SI_NB_RESULTS 指向 results.jsonl")
        return runs[-1]
    base = ROOT_DIR / "results" / "notebook"
    dataset = os.environ.get("DATASET_PATH", "").strip() or _env("DATASET", "")
    if dataset:
        runs = sorted(base.glob(f"{Path(dataset).stem}/runs/*/results.jsonl"))  # 目录名是时间戳，按名字排序即按时间
        if not runs:
            raise FileNotFoundError(f"题集 {Path(dataset).stem} 还没有运行结果：{base / Path(dataset).stem / 'runs'} 下没有 "
                                    "results.jsonl。先运行 batch_qa，或设 RESULTS_PATH / SI_NB_RESULTS 指向 results.jsonl")
        return runs[-1]
    runs = sorted(base.glob("*/runs/*/results.jsonl"), key=lambda p: p.stat().st_mtime)
    if not runs:
        raise FileNotFoundError(f"{base} 下没有找到任何 results.jsonl。先运行 batch_qa，或设 RESULTS_PATH / SI_NB_RESULTS")
    return runs[-1]


INPUT_PATH = locate_results().resolve()
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"输入文件不存在：{INPUT_PATH}")
EVAL_DIR = INPUT_PATH.parent / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_RESULTS = EVAL_DIR / "eval_results.jsonl"
EVAL_SUMMARY = EVAL_DIR / "eval_summary.md"
EVAL_CSV = EVAL_DIR / "eval.csv"
EVAL_XLSX = EVAL_DIR / "eval.xlsx"
# deepeval 的本地状态目录默认是 cwd 下的 .deepeval，固定到 eval/ 里，不在仓库根或 Databricks 工作目录里乱写
os.environ.setdefault("DEEPEVAL_CACHE_FOLDER", str(EVAL_DIR / ".deepeval"))

print(f"输入文件 : {INPUT_PATH}")
print(f"输出目录 : {EVAL_DIR}")
print(f"通过阈值 : {PASS_THRESHOLD}，并发 {JUDGE_CONCURRENCY}，单次超时 {JUDGE_TIMEOUT:g}s")

## 裁判模型与评测指标

- `OpenAIJudge` 是 deepeval 的 `DeepEvalBaseLLM` 子类，包一个 OpenAI 兼容客户端（`base_url` / `api_key` / 模型名来自上面的配置，带 `LLM_EXTRA_HEADERS`）。
  deepeval 调用 `generate(prompt, schema=<pydantic 模型>)` 要结构化输出：先用 `response_format={"type": "json_object"}`（端点不支持就去掉重试），
  再按 schema 解析；解析失败就从文本里截取 `{...}` 再解析；还不行就返回原文，交给 deepeval 自己解析。
- 指标是一个 `GEval`（名字 `答案正确性`，`evaluation_params` = 问题 / 期望答案 / 实际回答），判定标准写在 `EVALUATION_STEPS` 里，可按需修改。
  裁判输出 0~10 分，deepeval 归一化到 0~1；`≥ PASS_THRESHOLD` 判为通过。

In [ ]:
from deepeval.metrics import GEval
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase

try:
    from deepeval.test_case import SingleTurnParams as _Params  # deepeval 4.x 的新名字
except ImportError:  # 旧版 deepeval
    from deepeval.test_case import LLMTestCaseParams as _Params
import openai
from pydantic import BaseModel, ValidationError

EVALUATION_STEPS = [
    "以期望答案为准，找出实际回答给出的最终结论（最终数值或最终表述），判断它与期望答案是否一致。",
    "数值允许单位换算与合理四舍五入：如 18.1 亿 = 1,810,000,000 = 1.81B，38% = 0.38；只要数值等价就算正确。",
    "实际回答里出现了正确数字、但最终结论是另一个数，或前后自相矛盾、给出多个互相冲突的答案，算错误。",
    "实际回答额外多说的内容（解释、出处、计算过程）不扣分，只要最终结论正确就算正确。",
    "实际回答明确表示无法找到、无法回答或信息不足，而期望答案有具体内容时，算错误。",
    "期望答案是文字（非数值）时，按语义是否一致判断，不要求字面一致。",
    "完全正确给 10 分，结论错误给 0 分；只有在部分正确（如多项答案只答对一部分）时才给中间分。理由用中文简要说明。",
]


def _json_block(text: str) -> str | None:
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1] if start != -1 and end > start else None


def _parse(text: str, schema: type[BaseModel] | None) -> Any:
    """按 schema 解析；失败就截取文本里的 {...} 再解析；还不行返回原文（deepeval 会自己再尝试解析）。"""
    if schema is None:
        return text
    for candidate in (text, _json_block(text)):
        if candidate:
            try:
                return schema.model_validate_json(candidate)
            except (ValidationError, ValueError):
                pass
    return text


class OpenAIJudge(DeepEvalBaseLLM):
    """OpenAI 兼容 Chat Completions 端点上的裁判模型。"""

    def __init__(self, model: str, base_url: str | None, api_key: str | None):
        self.model_name, self.base_url, self.api_key = model, base_url or None, api_key or "EMPTY"
        self.json_mode = True  # 端点不支持 response_format 时自动关掉
        self._async_client = None
        super().__init__(model)

    def load_model(self) -> openai.OpenAI:
        return openai.OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=JUDGE_TIMEOUT,
                             default_headers=LLM_EXTRA_HEADERS or None)

    def _kwargs(self, prompt: str, schema: type[BaseModel] | None) -> dict[str, Any]:
        kwargs: dict[str, Any] = {"model": self.model_name, "messages": [{"role": "user", "content": prompt}]}
        if JUDGE_TEMPERATURE is not None:
            kwargs["temperature"] = JUDGE_TEMPERATURE
        if schema is not None and self.json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        return kwargs

    def generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
        kwargs = self._kwargs(prompt, schema)
        try:
            resp = self.model.chat.completions.create(**kwargs)
        except openai.BadRequestError:
            if "response_format" not in kwargs:
                raise
            self.json_mode = False  # 端点不支持 JSON 模式：去掉重试，之后都不再带
            kwargs.pop("response_format")
            resp = self.model.chat.completions.create(**kwargs)
        return _parse(resp.choices[0].message.content or "", schema)

    async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
        if self._async_client is None:
            self._async_client = openai.AsyncOpenAI(base_url=self.base_url, api_key=self.api_key, timeout=JUDGE_TIMEOUT,
                                                    default_headers=LLM_EXTRA_HEADERS or None)
        kwargs = self._kwargs(prompt, schema)
        try:
            resp = await self._async_client.chat.completions.create(**kwargs)
        except openai.BadRequestError:
            if "response_format" not in kwargs:
                raise
            self.json_mode = False
            kwargs.pop("response_format")
            resp = await self._async_client.chat.completions.create(**kwargs)
        return _parse(resp.choices[0].message.content or "", schema)

    def get_model_name(self) -> str:
        return self.model_name


class FakeJudge(DeepEvalBaseLLM):
    """仅供自测（SI_NB_JUDGE_FAKE=1）：不联网，按期望答案里的数字 / 文字是否出现在回答里给 9 分或 1 分。"""

    def __init__(self):
        super().__init__("fake-judge")

    def load_model(self) -> "FakeJudge":
        return self

    @staticmethod
    def _section(prompt: str, title: str) -> str:
        m = re.search(rf"{title}:\n(.*?) \n\n", prompt, re.DOTALL)
        return m.group(1) if m else ""

    @staticmethod
    def _numbers(text: str) -> set[float]:
        return {float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}

    def generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
        expected, actual = self._section(prompt, "Expected Output"), self._section(prompt, "Actual Output")
        wanted = self._numbers(expected)
        hit = wanted <= self._numbers(actual) if wanted else expected.strip().casefold() in actual.casefold()
        text = json.dumps({"reason": f"（假裁判）期望答案{'出现在' if hit else '没有出现在'}回答里", "score": 9 if hit else 1},
                          ensure_ascii=False)
        return _parse(text, schema)

    async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
        return self.generate(prompt, schema)

    def get_model_name(self) -> str:
        return "fake-judge"


def judge_settings() -> tuple[str, str, str]:
    """(模型名, base_url, api_key)：OPENAI_* 优先；没设 OPENAI_MODEL 时用 batch_qa 的 SUPERINDEX_*（openai/ 前缀）。"""
    model = os.environ.get("OPENAI_MODEL", "").strip()
    base_url = os.environ.get("OPENAI_BASE_URL", "").strip()
    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    chat_model = os.environ.get("SUPERINDEX_CHAT_MODEL", "").strip()
    if not model and chat_model.startswith("openai/"):
        model = chat_model.removeprefix("openai/")
        base_url = os.environ.get("SUPERINDEX_BASE_URL", "").strip() or base_url
        api_key = os.environ.get("SUPERINDEX_API_KEY_OVERRIDE", "").strip() or api_key
    return JUDGE_MODEL or model, base_url, api_key


def make_judge() -> DeepEvalBaseLLM:
    if JUDGE_FAKE:
        return FakeJudge()
    model, base_url, api_key = judge_settings()
    if not model:
        raise RuntimeError("未配置裁判模型：请设置 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY（.env 或「模型配置」cell 的 "
                           "LLM_CONFIG），或配置项 JUDGE_MODEL / 环境变量 SI_NB_JUDGE_MODEL")
    return OpenAIJudge(model, base_url, api_key)


def make_metric(judge: DeepEvalBaseLLM) -> GEval:
    """每题新建一个指标（GEval 实例会记录本次的 score / reason，不能在线程间共用）。"""
    return GEval(name="答案正确性", evaluation_steps=EVALUATION_STEPS,
                 evaluation_params=[_Params.INPUT, _Params.EXPECTED_OUTPUT, _Params.ACTUAL_OUTPUT],
                 threshold=PASS_THRESHOLD, model=judge, async_mode=False, verbose_mode=False)


_model, _base_url, _ = judge_settings()
print("裁判模型 : " + ("fake-judge（SI_NB_JUDGE_FAKE=1，仅供自测）" if JUDGE_FAKE else
                      f"{_model or '（未配置）'}，地址 {_base_url or '（未设，用 OpenAI 官方）'}"))  # 不打印 key

## 读取结果、筛出要评测的题

`results.jsonl` 每个 `id` 取最后一条记录（与 `superindex.batch.read_results` 一致）。没有 `expected`、或 `batch_qa` 没成功（有 `error`、`status` 为 `skipped_no_text_layer` 等）的题标为 `not_evaluated`；
`eval_results.jsonl` 里已有结论（`evaluated` / `not_evaluated`）且问题、期望答案、回答都没变的题跳过。

In [ ]:
def read_jsonl(path: Path) -> dict[str, dict[str, Any]]:
    """每个 id 的最后一条记录（保持首次出现的顺序）。"""
    records: dict[str, dict[str, Any]] = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                records[str(rec.get("id"))] = rec
    return records


def skip_reason(r: dict[str, Any]) -> str | None:
    """不送裁判的原因；None = 要评测。"""
    if r.get("status"):
        return f"batch_qa 未回答：{r['status']}" + (f"（{r['error']}）" if r.get("error") else "")
    if r.get("error"):
        return f"batch_qa 出错：{r['error']}"
    if r.get("expected") in (None, ""):
        return "没有期望答案（expected）"
    return None


def base_row(r: dict[str, Any]) -> dict[str, Any]:
    return {"id": str(r.get("id")), "question": r.get("question"), "expected": r.get("expected"),
            "answer": r.get("answer"), "doc": r.get("doc")}


inputs = read_jsonl(INPUT_PATH)
previous = read_jsonl(EVAL_RESULTS)
done = {k for k, e in previous.items() if e.get("eval_status") in ("evaluated", "not_evaluated") and k in inputs
        and all(e.get(f) == inputs[k].get(f) for f in ("question", "expected", "answer"))}
todo = [r for k, r in inputs.items() if k not in done]
n_skip = sum(1 for r in todo if skip_reason(r))
print(f"共 {len(inputs)} 题：已有结论 {len(done)}，本次处理 {len(todo)}（送裁判 {len(todo) - n_skip}，不评测 {n_skip}）")

## 逐题评测

`JUDGE_CONCURRENCY` 道题同时评测；每题完成立即追加写入 `eval_results.jsonl`，并打印 `[序号/总数] id 得分 通过/不通过 耗时`。单题出错（如超时、裁判输出无法解析）只记 `eval_status=error`，不中断整批，下次运行会重评。

In [ ]:
lock = threading.Lock()
finished = [0]
judge = make_judge() if any(not skip_reason(r) for r in todo) else None  # 全是不评测的题时不需要模型
JUDGE_NAME = judge.get_model_name() if judge else None


def evaluate_one(r: dict[str, Any]) -> None:
    t0 = time.perf_counter()
    row = {**base_row(r), "eval_score": None, "eval_pass": None, "eval_reason": None,
           "eval_status": "not_evaluated", "judge_model": None, "elapsed_s": 0.0}
    reason = skip_reason(r)
    if reason:
        row["eval_reason"] = reason
    else:
        row["judge_model"] = JUDGE_NAME
        try:
            metric = make_metric(judge)
            metric.measure(LLMTestCase(input=str(r.get("question") or ""), actual_output=str(r.get("answer") or ""),
                                       expected_output=str(r.get("expected"))), _show_indicator=False)
            row.update(eval_score=round(float(metric.score), 4), eval_pass=bool(metric.success),
                       eval_reason=metric.reason, eval_status="evaluated")
        except Exception as exc:  # noqa: BLE001 - 单题失败只记下来，整批继续
            row.update(eval_reason=f"{type(exc).__name__}: {exc}", eval_status="error")
        row["elapsed_s"] = round(time.perf_counter() - t0, 2)
    with lock:  # 每题完成立即追加写入
        with EVAL_RESULTS.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        finished[0] += 1
        status = row["eval_status"]
        verdict = (f"{row['eval_score']:.2f} {'通过' if row['eval_pass'] else '不通过'}" if status == "evaluated" else
                   f"ERROR {row['eval_reason']}" if status == "error" else f"不评测：{row['eval_reason']}")
        print(f"[{finished[0]}/{len(todo)}] {row['id']}  {verdict}  {row['elapsed_s']:.1f}s", flush=True)


t0 = time.time()
if todo:
    with ThreadPoolExecutor(max_workers=max(1, JUDGE_CONCURRENCY)) as pool:
        list(pool.map(evaluate_one, todo))
wall = time.time() - t0
print(f"本次处理 {len(todo)} 题，用时 {wall:.1f}s")

## 汇总与导出

按 `results.jsonl` 的题目顺序读出每题最新的评测结果，生成 `eval_summary.md`、`eval.csv`（utf-8-sig）和 `eval.xlsx`。准确率 = 通过数 / 已评测数（不含未评测和报错的题）。

In [ ]:
latest = read_jsonl(EVAL_RESULTS)
rows = [latest[k] for k in inputs if k in latest]
evaluated = [r for r in rows if r["eval_status"] == "evaluated"]
passed = sum(1 for r in evaluated if r["eval_pass"])
n_not = sum(1 for r in rows if r["eval_status"] == "not_evaluated")
n_err = sum(1 for r in rows if r["eval_status"] == "error")
accuracy = passed / len(evaluated) if evaluated else None
acc_text = f"{accuracy:.1%}（{passed}/{len(evaluated)}）" if evaluated else "无（没有已评测的题）"
judges = sorted({r["judge_model"] for r in evaluated if r.get("judge_model")})


def _cell(text: Any, limit: int = 80) -> str:
    s = " ".join(str("" if text is None else text).split()).replace("|", "\\|")
    return s if len(s) <= limit else s[: limit - 1] + "…"


lines = [
    "# 答案评测（deepeval GEval：答案正确性）", "",
    f"- 时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"- 输入：`{INPUT_PATH}`",
    f"- 裁判模型：{', '.join(judges) or '-'}；通过阈值 {PASS_THRESHOLD}",
    f"- **准确率：{acc_text}**",
    f"- 题数 {len(rows)}：已评测 {len(evaluated)}，未评测 {n_not}，报错 {n_err}", "",
    "| ID | 问题 | 得分 | 通过 | 理由摘要 |", "|---|---|---|---|---|",
]
for r in rows:
    score_text = f"{r['eval_score']:.2f}" if r["eval_score"] is not None else "-"
    pass_text = {"evaluated": "✓" if r["eval_pass"] else "✗", "error": "报错", "not_evaluated": "未评测"}[r["eval_status"]]
    lines.append(f"| {_cell(r['id'], 40)} | {_cell(r['question'], 60)} | {score_text} | {pass_text} | {_cell(r['eval_reason'], 120)} |")
EVAL_SUMMARY.write_text("\n".join(lines) + "\n", encoding="utf-8")

df = pd.DataFrame([{**r, "doc": "; ".join(r["doc"]) if isinstance(r.get("doc"), list) else r.get("doc")} for r in rows])
df.to_csv(EVAL_CSV, index=False, encoding="utf-8-sig")
try:
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

    def clean_cell(v: Any) -> Any:
        return ILLEGAL_CHARACTERS_RE.sub("", v)[:32767] if isinstance(v, str) else v  # xlsx 不接受控制字符，单元格最长 32767 字符

    # DataFrame.map 需要 pandas>=2.1；更旧的版本退回 applymap（这样不必升级集群预装的 pandas）
    (df.map(clean_cell) if hasattr(df, "map") else df.applymap(clean_cell)).to_excel(EVAL_XLSX, index=False)
    xlsx_path = EVAL_XLSX
except ImportError:
    xlsx_path = None
    print("未安装 openpyxl，跳过 eval.xlsx（本地 uv sync --group notebook；Databricks 上 %pip install openpyxl）")
display(df)

In [ ]:
print(f"准确率 : {acc_text}；未评测 {n_not}，报错 {n_err}，共 {len(rows)} 题")
print("产出文件：")
for path in (EVAL_RESULTS, EVAL_SUMMARY, EVAL_CSV, xlsx_path):
    if path is not None:
        print(f"  {path}")